# 03 — GRASP Model

GRASP (GRaph cAre-path Sequential Pattern) — graph-based model for EHR prediction.
Captures co-occurrence relationships between diagnoses.

In [ ]:
from pyhealth_enterprise.datasets.synthetic import SyntheticEHRDataset
from pyhealth.tasks import readmission_prediction_mimic3_fn
from pyhealth.datasets import split_by_patient, get_dataloader
from pyhealth.models import GRASP
from pyhealth.trainer import Trainer

ds = SyntheticEHRDataset(); ds.load()
task_dataset = ds.dataset.set_task(readmission_prediction_mimic3_fn)
train, val, test = split_by_patient(task_dataset, [0.8, 0.1, 0.1])
train_loader = get_dataloader(train, batch_size=32, shuffle=True)
val_loader   = get_dataloader(val,   batch_size=32, shuffle=False)
test_loader  = get_dataloader(test,  batch_size=32, shuffle=False)

model = GRASP(
    dataset=task_dataset,
    feature_keys=['conditions', 'drugs'],
    label_key='readmission',
    mode='binary',
)
trainer = Trainer(model=model, metrics=['pr_auc', 'roc_auc', 'f1'])
trainer.train(train_dataloader=train_loader, val_dataloader=val_loader,
              epochs=50, monitor='pr_auc')
print(trainer.evaluate(test_loader))